In [1]:
#cell1
# Setup imports

import json
import re
import unicodedata
from pathlib import Path

import pandas as pd
from google.colab import drive
from IPython.display import display

In [2]:
#cell2
# Mount Google Drive silently

import io
import contextlib

with contextlib.redirect_stdout(io.StringIO()):
    drive.mount("/content/drive", force_remount=False)

In [3]:
# cell 3
# Define PyRAG answer file paths, dataset names, and model names.
# This cell only changes the input files and does not change the F1 logic.

BASE_DIR = Path("/content/drive/MyDrive/final_project/pyrag/answers")

METHOD_NAME = "pyrag"

ANSWER_FILES = [
    {
        "dataset": "2wikimultihopqa",
        "model": "gemma4",
        "file_name": "2wikimultihopqa_gemma4_answers.json",
        "file_path": BASE_DIR / "2wikimultihopqa_gemma4_answers.json",
    },
    {
        "dataset": "hotpotqa",
        "model": "gemma4",
        "file_name": "hotpotqa_gemma4_answers.json",
        "file_path": BASE_DIR / "hotpotqa_gemma4_answers.json",
    },
    {
        "dataset": "2wikimultihopqa",
        "model": "gpt_oss_120b",
        "file_name": "2wikimultihopqa_gpt_oss_120b_answers.json",
        "file_path": BASE_DIR / "2wikimultihopqa_gpt_oss_120b_answers.json",
    },
    {
        "dataset": "hotpotqa",
        "model": "gpt_oss_120b",
        "file_name": "hotpotqa_gpt_oss_120b_answers.json",
        "file_path": BASE_DIR / "hotpotqa_gpt_oss_120b_answers.json",
    },
    {
        "dataset": "2wikimultihopqa",
        "model": "qwen3.5",
        "file_name": "2wikimultihopqa_qwen3.5_answers.json",
        "file_path": BASE_DIR / "2wikimultihopqa_qwen3.5_answers.json",
    },
    {
        "dataset": "hotpotqa",
        "model": "qwen3.5",
        "file_name": "hotpotqa_qwen3.5_answers.json",
        "file_path": BASE_DIR / "hotpotqa_qwen3.5_answers.json",
    },
]

print("METHOD_NAME:", METHOD_NAME)
print("BASE_DIR:", BASE_DIR)

for file_info in ANSWER_FILES:
    print(file_info["file_name"], "=>", file_info["file_path"])
    assert file_info["file_path"].is_file(), f"Missing answer file: {file_info['file_path']}"

print("All PyRAG answer files exist.")

METHOD_NAME: pyrag
BASE_DIR: /content/drive/MyDrive/final_project/pyrag/answers
2wikimultihopqa_gemma4_answers.json => /content/drive/MyDrive/final_project/pyrag/answers/2wikimultihopqa_gemma4_answers.json
hotpotqa_gemma4_answers.json => /content/drive/MyDrive/final_project/pyrag/answers/hotpotqa_gemma4_answers.json
2wikimultihopqa_gpt_oss_120b_answers.json => /content/drive/MyDrive/final_project/pyrag/answers/2wikimultihopqa_gpt_oss_120b_answers.json
hotpotqa_gpt_oss_120b_answers.json => /content/drive/MyDrive/final_project/pyrag/answers/hotpotqa_gpt_oss_120b_answers.json
2wikimultihopqa_qwen3.5_answers.json => /content/drive/MyDrive/final_project/pyrag/answers/2wikimultihopqa_qwen3.5_answers.json
hotpotqa_qwen3.5_answers.json => /content/drive/MyDrive/final_project/pyrag/answers/hotpotqa_qwen3.5_answers.json
All PyRAG answer files exist.


In [4]:
#cell4
# Normalize text and tokenize answers

def normalize_text(text):
    """
    Basic normalization for token-level comparison.
    Lowercase, remove punctuation, and normalize spaces.
    """
    if text is None:
        text = ""

    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.casefold()

    chars = []
    for ch in text:
        if unicodedata.category(ch).startswith("P"):
            chars.append(" ")
        else:
            chars.append(ch)

    text = "".join(chars)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text):
    """
    Convert answer text to a set of tokens.
    This keeps the same set-overlap logic as the previous notebook.
    """
    normalized = normalize_text(text)
    if not normalized:
        return set()
    return set(normalized.split())

In [5]:
#cell5
# Compute Precision, Recall, and F1 for one example

def compute_token_f1(predicted_answer, ground_truth_answer):
    """
    Compute token-level Precision, Recall, and F1.
    """
    pred_tokens = tokenize(predicted_answer)
    gt_tokens = tokenize(ground_truth_answer)

    if len(pred_tokens) == 0 and len(gt_tokens) == 0:
        return 1.0, 1.0, 1.0

    if len(pred_tokens) == 0 or len(gt_tokens) == 0:
        return 0.0, 0.0, 0.0

    overlap = pred_tokens.intersection(gt_tokens)

    precision = len(overlap) / len(pred_tokens)
    recall = len(overlap) / len(gt_tokens)

    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = (2 * precision * recall) / (precision + recall)

    return precision, recall, f1

In [6]:
#cell6
# Load one JSON file and compute row-level scores

def load_json_file(file_path):
    """
    Load a JSON answer file.
    """
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError(f"Expected a list of records in: {file_path}")

    return data


def evaluate_file(dataset_name, model_name, file_name, file_path):
    """
    Evaluate all questions for one dataset-model file.
    """
    data = load_json_file(file_path)
    rows = []

    for idx, item in enumerate(data):
        gt = item.get("gt", "")
        response = item.get("response", "")
        question_type = item.get("type", "unknown")

        precision, recall, f1 = compute_token_f1(
            predicted_answer=response,
            ground_truth_answer=gt
        )

        rows.append({
            "dataset": dataset_name,
            "model": model_name,
            "file_name": file_name,
            "row_index": idx,
            "source_index": item.get("source_index", idx),
            "type": question_type if question_type is not None else "unknown",
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "question": item.get("question", ""),
            "gt": gt,
            "response": response,
        })

    return pd.DataFrame(rows)

In [7]:
#cell7
# Evaluate all files while keeping dataset, model, and file identity separated

def evaluate_all_files(answer_files):
    """
    Evaluate every JSON file and combine row-level results.
    Each row keeps dataset, model, and file_name to prevent mixing results.
    """
    all_dfs = []

    for file_info in answer_files:
        file_df = evaluate_file(
            dataset_name=file_info["dataset"],
            model_name=file_info["model"],
            file_name=file_info["file_name"],
            file_path=file_info["file_path"],
        )
        all_dfs.append(file_df)

    return pd.concat(all_dfs, ignore_index=True)

In [8]:
# cell 8
# Build overall and per-type summaries for each separate PyRAG answer file.
# This cell keeps the original macro F1 calculation logic unchanged.
# Only the display/order metadata is updated for the PyRAG files.

def build_summary(scores_df):
    """
    Create overall and per-type macro summaries.
    Results are grouped by dataset, model, and file_name so files do not get mixed.
    """
    group_cols = ["dataset", "model", "file_name"]

    overall_df = (
        scores_df
        .groupby(group_cols, as_index=False)
        .agg(
            n_questions=("f1", "size"),
            precision_macro=("precision", "mean"),
            recall_macro=("recall", "mean"),
            f1_macro=("f1", "mean"),
        )
    )
    overall_df["type"] = "overall"

    type_df = (
        scores_df
        .groupby(group_cols + ["type"], as_index=False)
        .agg(
            n_questions=("f1", "size"),
            precision_macro=("precision", "mean"),
            recall_macro=("recall", "mean"),
            f1_macro=("f1", "mean"),
        )
    )

    summary_df = pd.concat([overall_df, type_df], ignore_index=True)

    summary_df["f1_percent"] = summary_df["f1_macro"] * 100

    dataset_order = ["2wikimultihopqa", "hotpotqa"]
    model_order = ["gemma4", "gpt_oss_120b", "qwen3.5"]
    file_order = [file_info["file_name"] for file_info in ANSWER_FILES]

    summary_df["dataset"] = pd.Categorical(
        summary_df["dataset"],
        categories=dataset_order,
        ordered=True
    )

    summary_df["model"] = pd.Categorical(
        summary_df["model"],
        categories=model_order,
        ordered=True
    )

    summary_df["file_name"] = pd.Categorical(
        summary_df["file_name"],
        categories=file_order,
        ordered=True
    )

    summary_df["type_sort"] = summary_df["type"].apply(
        lambda x: "000_overall" if x == "overall" else str(x)
    )

    summary_df = (
        summary_df
        .sort_values(["file_name", "type_sort"])
        .drop(columns=["type_sort"])
        .reset_index(drop=True)
    )

    column_order = [
        "dataset",
        "model",
        "file_name",
        "type",
        "n_questions",
        "precision_macro",
        "recall_macro",
        "f1_macro",
        "f1_percent",
    ]
    summary_df = summary_df[column_order]

    numeric_cols = ["precision_macro", "recall_macro", "f1_macro", "f1_percent"]
    summary_df[numeric_cols] = summary_df[numeric_cols].round(6)

    return summary_df

In [9]:
#cell9
# Final output: separate detailed summary for every dataset-model file

scores_df = evaluate_all_files(ANSWER_FILES)
summary_df = build_summary(scores_df)

for file_name in summary_df["file_name"].unique():
    print("=" * 100)
    print(f"Results for file: {file_name}")
    print("=" * 100)

    file_summary = summary_df[summary_df["file_name"] == file_name].reset_index(drop=True)
    display(file_summary)

Results for file: 2wikimultihopqa_gemma4_answers.json


,dataset,model,file_name,type,n_questions,precision_macro,recall_macro,f1_macro,f1_percent
0,2wikimultihopqa,gemma4,2wikimultihopqa_gemma4_answers.json,overall,1000,0.686361,0.728170,0.692923,69.292325
1,2wikimultihopqa,gemma4,2wikimultihopqa_gemma4_answers.json,bridge_comparison,250,0.756051,0.760000,0.756100,75.610000
2,2wikimultihopqa,gemma4,2wikimultihopqa_gemma4_answers.json,comparison,250,0.960052,0.964000,0.960103,96.010256
3,2wikimultihopqa,gemma4,2wikimultihopqa_gemma4_answers.json,compositional,250,0.470906,0.610667,0.502337,50.233724
4,2wikimultihopqa,gemma4,2wikimultihopqa_gemma4_answers.json,inference,250,0.558435,0.578014,0.553153,55.315320


Results for file: hotpotqa_gemma4_answers.json


,dataset,model,file_name,type,n_questions,precision_macro,recall_macro,f1_macro,f1_percent
0,hotpotqa,gemma4,hotpotqa_gemma4_answers.json,overall,1000,0.683628,0.706694,0.673712,67.371196
1,hotpotqa,gemma4,hotpotqa_gemma4_answers.json,bridge,700,0.632521,0.660479,0.625594,62.559423
2,hotpotqa,gemma4,hotpotqa_gemma4_answers.json,comparison,300,0.802878,0.814528,0.785987,78.598667


Results for file: 2wikimultihopqa_gpt_oss_120b_answers.json


...

,dataset,model,file_name,type,n_questions,precision_macro,recall_macro,f1_macro,f1_percent
0,hotpotqa,gpt_oss_120b,hotpotqa_gpt_oss_120b_answers.json,overall,1000,0.734620,0.720734,0.710201,71.020092
1,hotpotqa,gpt_oss_120b,hotpotqa_gpt_oss_120b_answers.json,bridge,700,0.688874,0.675014,0.663207,66.320705
2,hotpotqa,gpt_oss_120b,hotpotqa_gpt_oss_120b_answers.json,comparison,300,0.841361,0.827412,0.819853,81.985328


Results for file: 2wikimultihopqa_qwen3.5_answers.json


,dataset,model,file_name,type,n_questions,precision_macro,recall_macro,f1_macro,f1_percent
0,2wikimultihopqa,qwen3.5,2wikimultihopqa_qwen3.5_answers.json,overall,1000,0.721577,0.763098,0.730670,73.067017
1,2wikimultihopqa,qwen3.5,2wikimultihopqa_qwen3.5_answers.json,bridge_comparison,250,0.768000,0.768000,0.768000,76.800000
2,2wikimultihopqa,qwen3.5,2wikimultihopqa_qwen3.5_answers.json,comparison,250,0.988000,0.988000,0.988000,98.800000
3,2wikimultihopqa,qwen3.5,2wikimultihopqa_qwen3.5_answers.json,compositional,250,0.536484,0.670476,0.570669,57.066855
4,2wikimultihopqa,qwen3.5,2wikimultihopqa_qwen3.5_answers.json,inference,250,0.593822,0.625914,0.596012,59.601212


Results for file: hotpotqa_qwen3.5_answers.json


,dataset,model,file_name,type,n_questions,precision_macro,recall_macro,f1_macro,f1_percent
0,hotpotqa,qwen3.5,hotpotqa_qwen3.5_answers.json,overall,1000,0.718024,0.713118,0.697741,69.774075
1,hotpotqa,qwen3.5,hotpotqa_qwen3.5_answers.json,bridge,700,0.672838,0.669254,0.653082,65.308232
2,hotpotqa,qwen3.5,hotpotqa_qwen3.5_answers.json,comparison,300,0.823458,0.815468,0.801944,80.194376


In [10]:
#cell10
# Optional comparison table for easier model comparison by dataset and type

comparison_df = (
    summary_df
    .pivot_table(
        index=["dataset", "type"],
        columns="model",
        values="f1_percent",
        aggfunc="first"
    )
    .reset_index()
)

comparison_df.columns.name = None

display(comparison_df)

/tmp/ipykernel_16941/3631904528.py:6: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  .pivot_table(


,dataset,type,gemma4,gpt_oss_120b,qwen3.5
0,2wikimultihopqa,bridge_comparison,75.610000,80.194872,76.800000
1,2wikimultihopqa,comparison,96.010256,96.490110,98.800000
2,2wikimultihopqa,compositional,50.233724,55.811972,57.066855
3,2wikimultihopqa,inference,55.315320,60.710985,59.601212
4,2wikimultihopqa,overall,69.292325,73.301985,73.067017
5,hotpotqa,bridge,62.559423,66.320705,65.308232
6,hotpotqa,comparison,78.598667,81.985328,80.194376
7,hotpotqa,overall,67.371196,71.020092,69.774075
